In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
import matplotlib.pyplot as plt

sample_dataset = datasets.ImageFolder(
    "../dataset/train",
    transform=train_transform
)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, ax in enumerate(axes.flat):
    image, label = sample_dataset[i]

    # Undo normalization just for visualization
    image = image.permute(1, 2, 0)
    image = image * torch.tensor([0.229, 0.224, 0.225]) + \
            torch.tensor([0.485, 0.456, 0.406])

    image = image.clamp(0, 1)

    ax.imshow(image)
    ax.set_title(sample_dataset.classes[label])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
train_data = datasets.ImageFolder(
    "../dataset/train",
    transform=train_transform
)

val_data= datasets.ImageFolder(
    "../dataset/train",
    transform=val_test_transform
)

test_data = datasets.ImageFolder(
    "../dataset/test",
    transform=val_test_transform
)

print("Classes:", train_data.classes)
print("Training images:", len(train_data))
print("Validation source images:", len(val_data))
print("Test images:", len(test_data))

In [ ]:
targets = np.array(
    train_data.targets
)

indices = np.arange(
    len(train_data)
)

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=targets    
)

print("Training samples:", len(train_indices))
print("Validation samples:", len(val_indices))

In [ ]:
train_subset = Subset(
    train_data,
    train_indices
)

val_subset = Subset(
    val_data,
    val_indices
)

print(
    "Train subset:",
    len(train_subset)
)

print(
    "Validation subset:",
    len(val_subset)
)

In [ ]:
train_loader= DataLoader(
    train_subset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size=16,
    shuffle=False
)

print("DataLoaders created.")

In [ ]:
from collections import Counter

train_labels = [
    train_data.targets[i]
    for i in train_indices
]

val_labels = [
    val_data.targets[i]
    for i in val_indices
]

print("TRAIN DISTRIBUTION")
print(Counter(train_labels))

print("\nVALIDATION DISTRIBUTION")
print(Counter(val_labels))

print("\nTEST DISTRIBUTION")
print(Counter(test_data.targets))

In [ ]:
images, labels = next(iter(train_loader))

print("Image shape:", images.shape)
print("Labels:", labels.tolist())
print("Unique labels:", torch.unique(labels).tolist())

In [ ]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("First 10 labels:", labels[:10])

In [ ]:
class BasicCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(BasicCNN, self).__init__() # This initializes the parent nn.Module class.

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv2d( input channels, output channels, kernel size, padding )
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
model = BasicCNN(num_classes=4).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

optimizer.zero_grad()

outputs = model(images)

loss = criterion(outputs, labels)

loss.backward()

In [ ]:
old_weight = model.features[0].weight.clone().detach()

optimizer.step()

change = (model.features[0].weight.detach() - old_weight).abs().mean().item()

print("Parameter change:", change)

In [ ]:
print(
    "Gradient mean:",
    model.features[0].weight.grad.abs().mean().item()
)


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):

    # ----- Training -----
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_accuracy = correct / total

    # ----- Validation -----
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss / val_total
    val_accuracy = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.4f}"
    )

In [ ]:
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predictions = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

In [ ]:
test_accuracy = accuracy_score(all_labels, all_predictions)
test_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Macro F1 Score: {test_f1:.4f}")

In [ ]:
print(classification_report(
    all_labels,
    all_predictions,
    target_names=train_data.classes
))

In [ ]:
cm = confusion_matrix(all_labels, all_predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=train_data.classes
)

fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("Experiment 1 - Basic CNN Confusion Matrix")
plt.tight_layout()
plt.show()